## Socrates / Running Example
Tracking knowledge from elicited data to rules: A simple example using the famous syllogism "All men are mortal;Socrates is a man; therefore, Socrates is mortal."

### 0. Import functions shared with the other domain examples

In [1]:
import sys
sys.path.append("..")
from yarrrmlHandler import YarrrmlHandler
from etc import export_sheets_to_csv
from rdf_etc import run_query_file, add_namespaces, show_rdf_graph

### 1. Export all Excel worksheets as csv files (YARRRML does not inherently support xlsx files)

In [2]:
for filename in ['claimGenerationNetwork', 'claimFormalizationNetwork']:
    export_sheets_to_csv(
        excel_file=filename + '.xlsx', 
        output_dir='csv'
)

Exported experts -> csv\experts.csv
Exported events -> csv\events.csv
Exported resources -> csv\resources.csv
Exported claims -> csv\claims.csv
Exported claims -> csv\claims.csv
Exported kEngineers -> csv\kEngineers.csv
Exported formalRepresentations -> csv\formalRepresentations.csv
Exported assessments -> csv\assessments.csv
Exported CQs -> csv\CQs.csv
Exported formalCQs -> csv\formalCQs.csv


### 2. Convert csv data to RDF representation using the YARRRML specification
The graph is visualized using [yfiles-graphs-for-jupter](https://www.yworks.com/products/yfiles-graphs-for-jupyter). Please refer to the official documentation for using the exploration features. The visualization is designed for networkx graphs, not RDF graphs in specific. If you want to explore the graph with a specialized tool, please export the graph by uncommenting the serialization contextmanager

In [3]:
from rdflib import Graph
from rdf_etc import convert_and_targets, add_inferred_provenance

try:
    result = YarrrmlHandler.run_pipeline(
        working_dir=".",
        yarrrml_file="socrates.yarrrml.yml",
        rml_file="socrates_rules.rml.ttl",
    )
    print("Pipeline executed successfully.")
except Exception as ex:
    print(f"Error: {ex}")

g =  Graph().parse(data=result.stdout, format="nt")
g = add_namespaces(g)
g = convert_and_targets(g)
g = add_inferred_provenance(g)

# Uncomment to export RDF graph
# with open("rdf_export.ttl", mode="w", encoding="utf8") as f:
#     f.write(g.serialize(format="turtle"))
    
show_rdf_graph(g)

Pipeline executed successfully.


GraphWidget(layout=Layout(height='800px', width='100%'))

### 3. Store conditional knowledge base as annotation page as snapshot for further processing

In [1]:
from etc import output_pretty_rdf

result = run_query_file(g, "get_problog_kb.rq") # or for first version: get_fol_kb.rq"
constructed_graph = Graph()
for triple in result:
    constructed_graph.add(triple)

constructed_graph = add_namespaces(constructed_graph)
g += constructed_graph
ttl = constructed_graph.serialize(format="turtle")
output_pretty_rdf(ttl)

ImportError: cannot import name 'output_pretty_rdf' from 'etc' (unknown location)

### 4. Extract applicable rules from the knowledge base and execute to check the competency question "Can the system infer that Socrates is mortal?"

In [35]:
import pandas as pd
from problog import get_evaluatable
from problog.program import PrologString

def execute_problog(lines: list[str]):
    program = "\n".join(lines)
    model = PrologString(program)
    return {str(k): v for k, v in get_evaluatable().create_from(model).evaluate().items()}

results =  run_query_file(g, "extract_rules_template.rq", {"knowledge_base":"ex:ProbLogKnowledgeBase"})

df = pd.DataFrame(
    [(str(row.item), str(row.bodyValue)) for row in results],
    columns=["Node", "Rule"]
)

display(df)

result = execute_problog(df.Rule.tolist())
print(f"The probability that \"Socrates is mortal\" is {result['mortal(socrates)']} ."  )

,Node,Rule
0,http://www.example.com/CQ_AllMenAreMortal_ProbLog,query(mortal(socrates)).
1,http://www.example.com/ProbLog_AllHumansAreMortal,mortal(X) :- human(X).
2,http://www.example.com/ProbLog_SocratesIsAHuman,human(socrates).


The probability that "Socrates is mortal" is 1.0 .


### 5. (Optional) Extend the basic example
The example introduced all the essential steps from recorded data to executable rules. The example was kept simple and took many short-cuts regarding data processing and transformation which is only possible as the data is kept to a minimum. Add additional data to the Excel spreadsheets and repeat the process. Do the outcomes change? Do the basic SPARQL queries still work and if not, how best to adjust them to keep them robust and future-proof for additional data? 